# Merge WP9 shards and run the G2 gate

Place `wp9_shard_00.json` through `wp9_shard_26.json` in one directory, then run this notebook. It rejects duplicate result rows, merges the long-format JSON, runs the prespecified G2 checker, and displays compact summaries.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = os.environ.get("WDCF_REPO_URL", "https://github.com/hugogobato/wasserstein-causal-forests.git")
REPO_DIR = Path("/content/wasserstein-causal-forests")
if not (REPO_DIR / "research/sim/merge_results.py").exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / "research"))
os.chdir(REPO_DIR)

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SHARD_DIR = Path("/content/drive/MyDrive/wasserstein-causal-forests/wp9_shards")
else:
    SHARD_DIR = Path("/content/wp9_shards")
SHARD_FILES = sorted(SHARD_DIR.glob("wp9_shard_*.json"))
if not SHARD_FILES:
    raise FileNotFoundError(f"No shard files found in {SHARD_DIR}")
print(f"Found {len(SHARD_FILES)} shard files")

In [ ]:
MERGED_PATH = SHARD_DIR / "wp9_merged.json"
merge_command = [sys.executable, "research/sim/merge_results.py", *map(str, SHARD_FILES), "--out", str(MERGED_PATH)]
subprocess.run(merge_command, check=True)
rows = json.loads(MERGED_PATH.read_text())
print("Merged rows:", len(rows))

In [ ]:
g2 = subprocess.run(
    [sys.executable, "research/sim/g2_checks.py", str(MERGED_PATH)],
    text=True, capture_output=True,
)
print(g2.stdout)
if g2.stderr:
    print(g2.stderr)
if g2.returncode == 0:
    print("G2 gate: PASS")
else:
    print("G2 gate did not pass. Inspect the report above; incomplete pilots are expected to return nonzero.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(rows)
metrics = df[df.metric != "runtime_seconds"].copy()
summary = (metrics.groupby(["dgp_id", "observation_regime", "method", "metric"])["value"]
           .agg(mean="mean", sd="std", reps="count").reset_index())
display(summary.head(60))

primary = {"D1": "ise_curve", "D3": "ise_curve", "D4": "rmse_functional_0", "D5": "worst_standardized_error", "D8": "worst_standardized_error"}
plot_data = metrics[metrics.apply(lambda r: primary.get(r.dgp_id) == r.metric, axis=1)]
for (dgp, regime), group in plot_data.groupby(["dgp_id", "observation_regime"]):
    means = group.groupby("method").value.mean().sort_values()
    plt.figure(figsize=(10, 4))
    means.plot.bar()
    plt.title(f"{dgp}, {regime}: {primary[dgp]}")
    plt.ylabel("mean metric, lower is better")
    plt.xticks(rotation=70, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Safe download fallback for the merged result and gate report.
g2_report = SHARD_DIR / "g2_report.txt"
g2_report.write_text(g2.stdout)
try:
    from google.colab import files
    files.download(str(MERGED_PATH))
    files.download(str(g2_report))
    print("Downloaded:", MERGED_PATH, g2_report)
except Exception as e:
    print("(Not on Colab / download skipped):", e)